# PathSnap — Logistic Regression (Akshansh Agrawal)

Runs the shared `pathsnap` preprocessing and the Logistic Regression model from this repository. **No Google Drive needed:** you upload `Student folder.zip` once per session.

**Run order:** set the settings cell → Runtime → Run all → choose the zip when asked.

1. Clone the repository branch and install `requirements.txt`.
2. Upload and unzip the dataset (train + student_test only).
3. Run the unit tests.
4. Build candidate and label CSVs with `python -m pathsnap.build` (route is never read for candidates).
5. Train: 5-fold GroupKFold by trip on 80 train trips, refit, one evaluation on the 10 student_test trips.
6. Inspect results, then download `outputs/` as a zip.

**Smoke mode first.** `SMOKE = True` uses 4 trips per partition and 2 folds (about 1–2 minutes). Set it to `False` for the full run.

**Expected runtime and memory (full run, standard Colab CPU, 2 cores, ~12 GB RAM):**
| Step | Time | Memory |
| --- | --- | --- |
| Build features (90 trips) | ~3–6 min | < 2 GB |
| Load CSVs (train ~220 MB, 1.36 M rows × 28 columns) | ~20 s | ~1.5 GB peak |
| 5-fold CV + refit + test | ~3–8 min | ~2 GB |

In [ ]:
# ======================= SETTINGS (edit only this cell) =======================
REPO_URL = "https://github.com/aditya-130906/pathsnap.git"
BRANCH   = "feature/shared-pipeline-colab"   # use "main" once this branch is merged
SMOKE    = True        # True: 4 trips per partition, 2 folds. False: full 80 + 10 trips, 5 folds
WORKDIR  = "/content"
# ==============================================================================
import os
REPO = f"{WORKDIR}/pathsnap"
DATA = f"{WORKDIR}/data"
FEATURES = f"{REPO}/outputs/features_smoke" if SMOKE else f"{REPO}/outputs/features"
OUT = f"{REPO}/outputs/logistic_regression_smoke" if SMOKE else f"{REPO}/outputs/logistic_regression"
print(REPO, FEATURES, OUT, sep="\n")

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1
!pip -q install -r requirements.txt

In [ ]:
# Upload Student folder.zip (contains train/ and student_test/). Evaluator trips are not part of it.
import zipfile, glob
if not glob.glob(f"{DATA}/**/student_test", recursive=True):
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(n for n in uploaded if n.lower().endswith(".zip"))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(DATA)
    os.remove(zip_name)
print([p for p in glob.glob(f"{DATA}/**", recursive=True) if p.endswith(("train", "student_test"))])

In [ ]:
# Unit tests + a real-data build/training-loop test on 3 trips per partition
os.environ["PATHSNAP_DATA"] = DATA
!python -m pytest tests/test_pathsnap.py -q

In [ ]:
smoke_flag = "--smoke 4" if SMOKE else ""
!python -m pathsnap.build --dataset "{DATA}" --output "{FEATURES}" {smoke_flag} --jobs 2

In [ ]:
folds_flag = "--folds 2" if SMOKE else ""
!python models/logistic_regression/train_logreg.py --features "{FEATURES}" --dataset "{DATA}" --output "{OUT}" {folds_flag}

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
s = json.load(open(f"{OUT}/summary.json"))
keys = ["cv_macro_f1", "cv_macro_f1_std", "cv_train_macro_f1", "cv_top1_accuracy", "student_test_macro_f1",
        "student_test_top1_accuracy", "candidate_miss_rate", "precision", "recall", "log_loss",
        "route_continuity", "training_time_s", "inference_time_s"]
display(pd.Series({k: s[k] for k in keys}).round(4).to_frame("logistic_regression"))
folds = pd.read_csv(f"{OUT}/cv_folds.csv"); display(folds.round(4))
coef = pd.read_csv(f"{OUT}/coefficients.csv").set_index("feature").coefficient.sort_values()
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(folds.fold, folds.train_macro_f1, "o-", label="train"); ax[0].plot(folds.fold, folds.val_macro_f1, "o-", label="validation")
ax[0].set_xlabel("fold"); ax[0].set_ylabel("Macro F1"); ax[0].set_title("GroupKFold Macro F1"); ax[0].legend()
coef.plot.barh(ax=ax[1], title="Standardised coefficients")
plt.tight_layout(); plt.savefig(f"{OUT}/lr_plots.png", dpi=120); plt.show()
print("SMOKE RUN: not model-selection evidence" if SMOKE else "FULL RUN")

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(f"{WORKDIR}/logistic_regression_outputs", "zip", OUT)
files.download(archive)